In [1]:
from go_ml.models.warmup_esm_finetune import ESMFinetune
from argparse import ArgumentParser
import transformers
parser = ArgumentParser()
parser = ESMFinetune.add_model_specific_args(parser)
hparams = parser.parse_args([])  # Use an empty list to avoid parsing command line arguments
import pickle
data_path = "/home/andrew/GO_interp/data/train_esm_datasets/"
with open(f"{data_path}/train_dataset.pkl", "rb") as f:
    train_dataset = pickle.load(f)
with open(f"{data_path}/val_dataset.pkl", "rb") as f:
    val_dataset = pickle.load(f)
tokenizer = transformers.AutoTokenizer.from_pretrained(hparams.model_name)
from go_ml.data_utils import prot_func_collate
hparams.num_classes = train_dataset[0]['labels'].shape[0]
hparams.num_train_steps = 10*len(train_dataset)
hparams.cls_warmup = True

In [2]:
import torch
from torch.utils.data import DataLoader
tokenizer = transformers.AutoTokenizer.from_pretrained(hparams.model_name)
dataloader_params = {"shuffle": True, "batch_size": 6, "collate_fn":prot_func_collate}
val_dataloader_params = {"shuffle": False, "batch_size": 12, "collate_fn":prot_func_collate}

train_loader = DataLoader(train_dataset, **dataloader_params)
val_loader = DataLoader(val_dataset, **val_dataloader_params)

import pytorch_lightning as pl
#load model from checkpoint
checkpoint_path = "/home/andrew/GO_interp/checkpoints/esm_finetune_fin.ckpt"
model = ESMFinetune.load_from_checkpoint(checkpoint_path).to(device="cuda:0")

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [1]:
import os, json, pickle
import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

from go_ml.models.warmup_esm_finetune import ESMFinetune
from go_ml.data_utils import *
from argparse import ArgumentParser
import transformers

parser = ArgumentParser()
parser = ESMFinetune.add_model_specific_args(parser)
hparams = parser.parse_args([])
print("got hparams", hparams, type(hparams))

import pickle
data_path = "/home/andrew/GO_interp/data/train_esm_datasets/"
with open(f"{data_path}/train_dataset.pkl", "rb") as f:
    train_dataset = pickle.load(f)
with open(f"{data_path}/val_dataset.pkl", "rb") as f:
    val_dataset = pickle.load(f)
tokenizer = transformers.AutoTokenizer.from_pretrained(hparams.model_name)
from go_ml.data_utils import prot_func_collate
hparams.num_classes = train_dataset[0]['labels'].shape[0]
hparams.num_train_steps = 10*len(train_dataset)
hparams.cls_warmup = False
hparams.learning_rate = 3e-5

checkpoint_path = "/home/andrew/GO_interp/checkpoints/esm_finetune-v3.ckpt"
model = ESMFinetune.load_from_checkpoint(checkpoint_path)
model.unfreeze_encoder()  # Unfreeze the encoder for fine-tuning

/home/andrew/anaconda3/envs/goint/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


got hparams Namespace(model_name='facebook/esm2_t33_650M_UR50D', max_length=1024, learning_rate=1.5e-05, gradient_checkpointing=False, gradient_clipping=1.0, weight_decay=0.01, cls_warmup=False) <class 'argparse.Namespace'>


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
tokenizer = transformers.AutoTokenizer.from_pretrained(hparams.model_name)
dataloader_params = {"shuffle": True, "batch_size": 2, "collate_fn": prot_func_collate}
val_dataloader_params = {"shuffle": False, "batch_size": 3, "collate_fn": prot_func_collate}

train_loader = DataLoader(train_dataset, **dataloader_params)
val_loader = DataLoader(val_dataset, **val_dataloader_params)

def check_model_nan(model):
    for name, param in model.named_parameters():
        # print(name, param.shape, param.requires_grad)
        if(param.isnan().any()):
            print(f"NaN detected in parameter {name}")
            print(f"Parameter shape: {param.shape}")
def check_model_grad_nan(model):
    for name, param in model.named_parameters():
        # print(name, param.shape, param.requires_grad)
        if param.requires_grad:
            # print(f"{name}: {param.grad.norm()}")
            if(not param.grad is None and param.grad.isnan().any()):
                print(f"NaN detected in parameter {name}")
                print(f"Parameter shape: {param.shape}")
#Add forward hook to model
def check_nan(module, input, output):
    if not input[0].isnan().any() and torch.isnan(output).any():
        print(f"NaN detected in output of {module.__class__.__name__}")
        print(f"Input shape: {input[0].shape}, Output shape: {output.shape}")

In [4]:
from itertools import islice
batches = list(islice(train_loader, 10))  # Get the first 10 batches
ind = 2
inputs, mask, y = batches[ind]['seq_ind'].to(device='cuda:0'), batches[ind]['mask'].to(device='cuda:0'), batches[ind]['labels'].to(device='cuda:0')
y_hat2 = model(inputs, mask)
loss_val = model.loss(y_hat2, y.float())
loss_val.backward()

In [5]:
model.model.embeddings.word_embeddings.weight.grad


tensor([[-4.9848e-07, -1.6403e-07,  1.0530e-06,  ..., -3.5198e-07,
         -4.7885e-07,  5.1398e-07],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 3.7924e-08, -1.5401e-07,  7.1039e-07,  ..., -2.5463e-07,
         -9.0012e-08, -9.6592e-08],
        ...,
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00]], device='cuda:0')

In [6]:
from torch.optim import AdamW
optimizer = AdamW(model.parameters(), lr=hparams.learning_rate)
optimizer.zero_grad()
y_hat = model.forward(inputs, mask)
loss_val = model.loss(y_hat, y.float())
loss_val.backward()
check_model_nan(model)
check_model_grad_nan(model)

In [7]:
for name, param in model.named_parameters():
    # print(name, param.shape, param.requires_grad)
    if param.requires_grad:
        if(not param.grad is None and param.grad.isnan().any()):
            print(f"NaN detected in parameter {name}")
            print(f"Parameter shape: {param.shape}")
        else:
            print(f"Parameter {name} is fine, shape: {param.shape}, requires grad {param.requires_grad}, grad norm: {param.grad.norm() if param.grad is not None else 'None'}")

Parameter model.embeddings.word_embeddings.weight is fine, shape: torch.Size([33, 1280]), requires grad True, grad norm: 0.002305174246430397
Parameter model.encoder.layer.0.attention.self.query.weight is fine, shape: torch.Size([1280, 1280]), requires grad True, grad norm: 0.00016257782408501953
Parameter model.encoder.layer.0.attention.self.query.bias is fine, shape: torch.Size([1280]), requires grad True, grad norm: 6.0878526710439473e-05
Parameter model.encoder.layer.0.attention.self.key.weight is fine, shape: torch.Size([1280, 1280]), requires grad True, grad norm: 0.0002052521303994581
Parameter model.encoder.layer.0.attention.self.key.bias is fine, shape: torch.Size([1280]), requires grad True, grad norm: 3.270021261414513e-05
Parameter model.encoder.layer.0.attention.self.value.weight is fine, shape: torch.Size([1280, 1280]), requires grad True, grad norm: 0.0008065992733463645
Parameter model.encoder.layer.0.attention.self.value.bias is fine, shape: torch.Size([1280]), require

In [8]:
optimizer.step()
check_model_nan(model)
check_model_grad_nan(model)

In [54]:
list(model.model.modules())
for module in model.model.modules():
    if isinstance(module, torch.nn.Linear):
        module.register_forward_hook(check_nan)

In [ ]:
y_hat = model.forward(inputs, mask)
loss_val = model.loss(y_hat, y.float())
loss_val.backward()

In [75]:
optimizer.param_groups[0]['lr']
optimizer.param_groups[0]['lr'] = 1e-12

In [68]:
optimizer.step()

In [62]:
optimizer.step()
print(y_hat)
y_hat = model.forward(inputs, mask)
print(y_hat)

tensor([[nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan]], device='cuda:0',
       grad_fn=<AddmmBackward0>)
tensor([[nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan]], device='cuda:0',
       grad_fn=<AddmmBackward0>)


tensor([[ -3.3387,  -0.0509,  -2.1071,  ..., -12.6085, -14.8765, -12.4033],
        [ -3.2275,  -1.9829,  -4.8958,  ..., -11.4557, -14.3781, -12.4342]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
tensor([[nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan]], device='cuda:0',
       grad_fn=<AddmmBackward0>)
